# M1 — Provenance & metadata: scenario testing

**Template for per-module scenario notebooks** (M2 watermarks, M3 forensics,
M4 detector will follow the same shape): synthetic fixtures first (deterministic,
CI-like), then curated real corpora, then *your* wild images, then the
**transport-degradation matrix** — which signals survive which handling.

Convention: commit this notebook WITH outputs — it's a run record.
Absolute paths everywhere; every cell survives a fresh kernel after Setup.

In [ ]:
# ── Setup — fresh-kernel-proof ──────────────────────────────────
!apt-get -qq install -y libimage-exiftool-perl > /dev/null
!git clone -q https://github.com/Waranika/AI-image-Checkers.git 2>/dev/null || echo "already cloned"
%cd /content/AI-image-Checkers
%pip install -q -e .

from ai_image_id.main import analyze_image

def report(path, label=""):
    """Compact human-readable evidence card for one image."""
    r = analyze_image(path)
    p = r.evidence.provenance
    print(f"┌─ {label or path}")
    print(f"│ verdict: {r.ai_verdict.value} ({r.confidence})")
    print(f"│ c2pa: present={p.c2pa_present} sig_valid={p.c2pa_signature_valid} "
          f"trusted={p.c2pa_signer_trusted} gen={p.c2pa_generator}")
    if p.c2pa_actions:            print(f"│ actions: {p.c2pa_actions}")
    if p.generation_params_tool:  print(f"│ gen-params: {p.generation_params_tool} model={p.generation_params_model}")
    if p.iptc_source_category:    print(f"│ iptc: {p.iptc_digital_source_type} → {p.iptc_source_category}")
    if p.camera_exif_present:     print(f"│ camera-exif: {p.camera_exif_fields} fields")
    print(f"└ notes: {r.notes}\n")
    return r

## 1 — Synthetic fixtures (deterministic)

Each M1 signal manufactured on demand — same fixtures as the pytest suite,
here for eyeballing the full evidence cards. Expected verdicts in comments.

In [ ]:
import numpy as np, subprocess
from pathlib import Path
from PIL import Image
from PIL.PngImagePlugin import PngInfo

FIX = Path("/content/fixtures"); FIX.mkdir(exist_ok=True)
rng = np.random.default_rng(7)
y, x = np.mgrid[0:512, 0:512]
base = np.clip(np.stack([120+60*np.sin(x/97), 100+50*np.cos(y/83),
                         90+40*np.sin((x+y)/71)], -1) + rng.normal(0,6,(512,512,3)),
               0, 255).astype(np.uint8)

# a) A1111 recipe chunk → likely 0.85, model extracted
p = FIX/"a1111.png"; Image.fromarray(base).save(p)
info = PngInfo(); info.add_text("parameters",
    "a castle on a hill\nNegative prompt: blurry\nSteps: 20, Sampler: Euler a, "
    "CFG scale: 7, Seed: 1234567, Model hash: 31e35c80fc, Model: sd_xl_base_1.0")
Image.open(p).save(p, pnginfo=info)

# b) ComfyUI workflow chunk → likely
p = FIX/"comfy.png"; Image.fromarray(base).save(p)
info = PngInfo(); info.add_text("prompt", '{"3": {"class_type": "KSampler"}}')
Image.open(p).save(p, pnginfo=info)

# c) IPTC vocabulary: ai / ai_composite / capture
for val in ["trainedAlgorithmicMedia", "compositeWithTrainedAlgorithmicMedia", "digitalCapture"]:
    p = FIX/f"iptc_{val[:9]}.jpg"; Image.fromarray(base).save(p, quality=92)
    subprocess.run(["exiftool", "-overwrite_original",
        f"-XMP-iptcExt:DigitalSourceType=http://cv.iptc.org/newscodes/digitalsourcetype/{val}",
        str(p)], check=True, capture_output=True)

# d) coherent camera block → note only, inconclusive
p = FIX/"camera.jpg"; Image.fromarray(base).save(p, quality=92)
subprocess.run(["exiftool", "-overwrite_original", "-Make=Canon", "-Model=EOS R5",
    "-ExposureTime=1/250", "-FNumber=2.8", "-ISO=400", str(p)], check=True, capture_output=True)

for f in sorted(FIX.glob("*")):
    report(str(f), f.name)

## 2 — C2PA corpus (real signed files, incl. tampered)

Expected: good manifests `sig_valid=True, trusted=False` (test certs — a config
fact, not a forgery signal); `E-*`/`X*` → `sig_valid=False`; `-A`/`-I` → no
manifest. NEW since the M1 upgrade: actions history extracted per file.

In [ ]:
!git clone -q --depth 1 https://github.com/c2pa-org/public-testfiles.git /content/public-testfiles 2>/dev/null || true

from pathlib import Path
corpus = Path("/content/public-testfiles/legacy/1.4/image/jpeg")
print(f"{'file':<42} {'verdict':<13} {'sig':<6} {'trust':<6} actions")
for p in sorted(corpus.glob("*.jpg")):
    r = analyze_image(p)
    v = r.evidence.provenance
    acts = ",".join(a.split(" ")[0].replace("c2pa.","") for a in v.c2pa_actions[:3])
    print(f"{p.name:<42} {r.ai_verdict.value:<13} {str(v.c2pa_signature_valid):<6} "
          f"{str(v.c2pa_signer_trusted):<6} {acts}")

## 3 — Your real images (wild validation)

Upload: OpenAI/DALL·E export (→ verified, now with actions history), Firefly,
Meta AI image (→ likely via IPTC), any A1111/ComfyUI image from civitai
(→ likely via gen-params — item 1's first wild confirmation), phone photo
(→ inconclusive + camera-exif note).

In [ ]:
from google.colab import files
up = files.upload()
for name in up:
    report(name)

## 4 — Transport-degradation matrix

The centerpiece: one *verified* image pushed through realistic transports;
which M1 signals survive each hop? Upload your metadata-intact OpenAI PNG
(or any C2PA-carrying AI image) when prompted.

Transports simulated: re-save PNG→JPEG (editor export), screenshot
(re-encode, all metadata gone), messenger (resize + Q70), explicit strip
(exiftool -all=).

In [ ]:
import shutil, subprocess
from PIL import Image
from pathlib import Path

up = files.upload()                      # the verified original
orig = Path(next(iter(up)))
HOP = Path("/content/hops"); HOP.mkdir(exist_ok=True)

def make_hops(src: Path) -> dict:
    hops = {"0-original": src}
    im = Image.open(src).convert("RGB")
    p = HOP/"1-resave.jpg";     im.save(p, quality=92);  hops["1-resave-jpg"] = p
    p = HOP/"2-screenshot.png"; im.save(p)               # fresh PNG ⇒ no chunks copied
    hops["2-screenshot"] = p
    p = HOP/"3-messenger.jpg";  im.resize((im.width//2, im.height//2)).save(p, quality=70)
    hops["3-messenger"] = p
    p = HOP/("4-stripped" + src.suffix); shutil.copy(src, p)
    subprocess.run(["exiftool", "-overwrite_original", "-all=", str(p)], capture_output=True)
    hops["4-exiftool-strip"] = p
    return hops

SIGNALS = ["c2pa", "gen_params", "iptc", "tool_fields", "camera_exif"]

def signal_row(path):
    r = analyze_image(path)
    v = r.evidence.provenance
    return {"c2pa": bool(v.c2pa_present and v.c2pa_signature_valid),
            "gen_params": bool(v.generation_params_tool),
            "iptc": bool(v.iptc_source_category),
            "tool_fields": any("=" in h and "c2pa" not in h for h in v.ai_metadata_hits),
            "camera_exif": v.camera_exif_present}, r.ai_verdict.value

hops = make_hops(orig)
print(f"{'transport':<20}" + "".join(f"{s:<13}" for s in SIGNALS) + "verdict")
for name, p in hops.items():
    sig, verdict = signal_row(p)
    print(f"{name:<20}" + "".join(f"{'✓' if sig[s] else '·':<13}" for s in SIGNALS) + verdict)

### Reading the matrix
Expected shape: every hop except `1-resave` (sometimes) kills everything —
the empirical demonstration that **M1 evidence is transport-fragile and
absence is non-evidence**. This table is blog/model-card material, and it's
the quantitative case for M2 (robust watermarks) and M5 (provenance recovery
via reverse search). Re-run per module as their scenario notebooks land:
watermarks should survive hops 1–3 far better; the classifier should survive
all of them with degraded confidence.